# Erdschlussmessung auswerten – Einzelmessung

Auswertung einer COMTRADE-Aufzeichnung eines Erdschlussversuchs im kompensierten Netz. Alle Berechnungen stecken im
Paket **`groundfault`** – das Notebook ist nur die Bedienoberfläche. Ausgangsfassung:
`Vorlagen/Messung_auswerten_DEFAULT.ipynb`.

**Ablauf**
1. **Konfiguration** – Kampagne, Datei, Zusatzimpedanz; eigene Grenzwerte, Zuschläge und Faktoren (groundfault hat
   keine). Auswertebereiche zunächst leer lassen.
2. **Datei öffnen** – Kanalrollen und Rohdaten-Diagnose prüfen.
3. **Effektivwertverlauf** – Zeiten für *stationären Fehler*, *Vorfehler* und *Nachfehler* ablesen
   und in Abschnitt 1 eintragen, dann das Notebook erneut ausführen (*Run All*). Kein Automatismus:
   aus allen 200-ms-Fenstern im stationären Bereich wird das Fenster mit dem **größten** Erdschlussreststrom bewertet.
4. **Auswertung** – Bewertungsgrößen mit den eigenen Messunsicherheitszuschlägen, Winkelverfahren, Plausibilitätsprüfungen, Spannungen
   vor/im/nach dem Fehler, ggf. Station und Trafo. An einer **Station**: Erdungs- und Berührungsspannung je Frequenz
   (direkt und fehlerbedingt), PEN-Ströme und Strom ins Ortsnetz (Z_E,Ortsnetz), Vergleich mit der
   **Erdungsmessung aus groundmeas** und **Reduktionsfaktor**.
5. **Signal-Explorer** – alle Kanäle ansehen: Effektivwert über der Zeit mit Harmonischen/THD im Mouse-over,
   Momentanwerte und Spektrum eines frei wählbaren Ausschnitts (prüfen, ob die Zahlen zur Kurve passen).
6. **Kontrollen** (optional) – Netzfrequenz, Spektren je Fenster, Vergleich mit den RMS-Werten des Messgeräts, V-Kurve.
7. **Export** – Ergebnisdatei (Grundlage für `Kampagne_auswerten.ipynb`), Tabellen (CSV/XLSX/DOCX/Markdown),
   Abbildungen (PDF/SVG/PNG, ohne Titel, A4-Satzspiegel) und der Signal-Explorer als HTML-Datei; Eintrag in groundmeas.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

LANGUAGE = "de"                # Sprache von Tabellen, Hinweisen und Abbildungen: "de" | "en"
try:
    import groundfault as gf
except ImportError:            # nicht installiert: Quellordner eines Klons verwenden (…/groundfault/src)
    for p in [Path.cwd(), *Path.cwd().parents]:    # ./src, ./groundfault/src, ./*/groundfault/src
        try:
            candidates = [p / "src", p / "groundfault" / "src", *sorted(p.glob("*/groundfault/src"))]
        except OSError:
            candidates = [p / "src", p / "groundfault" / "src"]
        hits = [c for c in candidates if (c / "groundfault" / "__init__.py").is_file()]
        if hits:
            sys.path.insert(0, str(hits[0]))
            break
    import groundfault as gf
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from groundfault import (campaign, corrections, demo, diagnostics, earthing, equivalent_source, explorer, i18n,
                         interactive, layout, legacy, register, report, rms, sites)

i18n.set_language(LANGUAGE)
# Arbeitsordner = dieser Ordner oder der erste Ordner darüber mit einem Kampagnenordner („01_Messkampagnen“
# oder „campaigns“); dessen Name gibt die Ordnerstruktur, ohne ihn gilt die Ordnerstruktur von LANGUAGE
WORK_FOLDER, LAYOUT = layout.find_work_folder(Path.cwd(), default=LANGUAGE)
WORK_FOLDER = Path(WORK_FOLDER)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: "%.4g" % x)
report.activate_style()        # Vorschau im Notebook = Schriftbild der exportierten Abbildungen


def show(df):
    """Tabelle mit Spaltennamen und Codes in der Ausgabesprache."""
    display(i18n.localize(df))


print("groundfault %s · Arbeitsordner: %s · Ordnerstruktur: %s · Schrift der Abbildungen: %s"
      % (gf.__version__, WORK_FOLDER, LAYOUT.language, report.used_font()))

## 1 · Konfiguration

In [ ]:
# ---------------------------------------------------------------- Messung
CAMPAIGN    = "2026-01-15_Demo"                     # Ordner im Kampagnenordner
FILE        = "1_Rohdaten/UW/UW_0_10_000kHz.cfg"     # .cfg relativ zum Kampagnenordner
MEASUREMENT = None                                   # Name für Exporte (None = Dateiname ohne Endung)
Z_FAULT     = 0.0                                    # Zusatzimpedanz im Fehlerzweig in Ω (UW: 0 / 2,5 / 5 – Station: None)
DEMO        = True                                   # Demo-Kampagne anlegen, falls sie fehlt (ändert nichts außerhalb)

# ---------------------------------------------------------------- Auswertebereiche in s (in Abschnitt 3 ablesen!)
# Demo-UW 0 Ω – Fehler von 2 s bis 6 s, Wattreststromerhöhung ab 4,6 s:
STEADY_STATE = (2.6, 4.4)   # stationärer Erdschluss – nach dem Einschwingen, vor einer Wattreststromerhöhung
PRE_FAULT    = (0.2, 1.8)   # erdschlussfreier Betrieb vor dem Fehler
POST_FAULT   = (7.0, 8.8)   # nach dem Abklingen der Verlagerungsspannung

# ---------------------------------------------------------------- Grenzwerte, Zuschläge und Faktoren – selbst festlegen
# groundfault enthält keine Grenzwerte oder Sicherheitsfaktoren: aus der Vorgabe des Netzbetreibers bzw. der geltenden
# Norm übernehmen. Nicht gesetzt (die Werte unten), werden die Teile, die sie brauchen, mit einem Hinweis übersprungen.
LIMITS            = ()      # Grenzwerte des Reststroms in A, z. B. (I_1, I_2): Hochrechnung der Stationsspannungen
U_TP              = None    # zulässige Berührungsspannung U_Tp in V (Bewertungsrelevanz des Reduktionsfaktors)
F_FACTOR          = None    # Faktor F der zulässigen Erdungsspannung F·U_Tp
REFERENCE_CURRENT = None    # Bezugsstrom in A für den Reduktionsfaktor (None = größter Wert von LIMITS)
UNCERTAINTY_I     = 0.0     # Messunsicherheitszuschlag m_I der Ströme als Anteil: I·(1 + m_I) (0.0 = keiner)
UNCERTAINTY_U     = 0.0     # Messunsicherheitszuschlag m_U der Spannungen als Anteil: U·(1 − m_U) (0.0 = keiner)
PQ_SAFETY_FACTOR  = 1.0     # Faktor auf das 95-%-Quantil einer PQ-Messung (1.0 = keiner)
R_TYPICAL         = None    # erwarteter Bereich (unten, oben) des Reduktionsfaktors bei 50 Hz, z. B. aus Erfahrung
# (LIMITS, U_TP, F_FACTOR, UNCERTAINTY_I/_U und PQ_SAFETY_FACTOR werden zu den Einstellungen ires_limits, u_tp,
#  f_factor, uncertainty_I/_U und pq_safety_factor und in der Ergebnisdatei gespeichert)

# ---------------------------------------------------------------- Auswerteparameter
SETTINGS = gf.Settings(
    T_win=0.200,                 # Fensterlänge in s (Schrittweite = Fensterlänge)
    harmonics=(1, 3, 5, 7),      # Ordnungen der Bewertungsgrößen
    uncertainty_I=UNCERTAINTY_I,  # vergrößert die Ströme
    uncertainty_U=UNCERTAINTY_U,  # verkleinert die Spannungen
    u_thd95_percent=None,        # 95-%-Quantil des THD (L-L) einer PQ-Messung in % (None = keine Skalierung)
    pq_safety_factor=PQ_SAFETY_FACTOR,
    correct_polarity=True,       # Iw < 0 → Stromzeiger um 180° drehen (verpolte Zange)
    ires_limits=LIMITS, u_tp=U_TP, f_factor=F_FACTOR,
)

# ---------------------------------------------------------------- Kanäle (nur bei Bedarf)
# Dauerhafte Korrekturen der Kampagne (falsche Wandlerübersetzung, Zange nur auf einem von mehreren Leitern,
# Polarität, Kanal ignorieren, Hauptkanal) gehören in 0_Notizen/Korrekturen.csv der Kampagne –
# sie werden automatisch angewendet und im Ergebnis dokumentiert. Hier nur Einstellungen für diesen Lauf:
APPLY_CORRECTIONS = True
ROLE_OVERRIDES = {}      # z. B. {"ISp22": None} = Kanal ignorieren, {"U1": "u_l1e"} = Rolle festlegen
MAIN_CHANNEL   = {}      # z. B. {"i_res": "IF2"} (Standard: IF vor IF1 vor IFr mit größtem Messbereich)
ALIGNMENT      = None    # Zeitversatz zwischen Subsystemen, z. B. dict(roles=["u_l1e", "u_l2e", "u_l3e"], dt_s=1.5)
FS_OVERRIDE    = None    # Abtastrate erzwingen (nur bei defekter Zeitbasis, siehe Diagnose)

# ---------------------------------------------------------------- Kontrollen & Export
RMS_FILE         = None      # RMS-Export des Messgeräts (Kontrolle), z. B. "1_Rohdaten/<Messort>/<Datei>_10_000Hz.cfg"
V_CURVE          = False   # Grundschwingungsverlauf/V-Kurve (Spulenverstellung während des Fehlers)
SHOW_ENVELOPE    = False   # zusätzlich Momentanwert-Hüllkurve der ganzen Aufzeichnung (interaktiv)
EXPORT_FORMATS   = ("pdf", "svg", "png")
RECORD_INTERVALS = True    # gewählte Bereiche im Messregister des Arbeitsordners dokumentieren (nicht mit DEMO = True)

# ---------------------------------------------------------------- Signal-Explorer und Erdungsmessung (Station)
SIGNAL_EXPLORER  = True    # Abschnitt 5: alle Kanäle ansehen (Mouse-over mit Effektivwert, Harmonischen, THD)
EXPLORER_HTML    = True    # Explorer zusätzlich als HTML-Datei exportieren (ohne Python im Browser zu öffnen)
GROUNDMEAS_DB    = str(WORK_FOLDER / "demo_groundmeas.db")   # Demo-Datenbank (mit der Demo-Kampagne angelegt)
GROUNDMEAS_LOCATION = None       # Ort in groundmeas; None = aus der Messorte-Datei der Kampagne bzw. dem Ordnernamen
GROUNDMEAS_MEASUREMENTS = None   # Vergleich: IDs der Messungen in groundmeas, z. B. [3, 21]; None = alle des Ortes
GROUNDMEAS_EXPORT = False        # True: als „staged_fault_test“ eintragen (nach Sicherungskopie); "replace": eigenen Eintrag erneuern
GROUNDMEAS_UNCERTAIN = "mark"    # unsichere fehlerbedingte Werte: "mark" (mit Vermerk) | "omit" (weglassen)
GROUNDMEAS_UNITS = "suitable"    # "suitable": je Größe gut lesbar (mV, mA …; groundmeas ≥ 1.6.1) | "base": V und A
GROUNDMEAS_DESCRIPTION = ""      # optional, vor der automatischen Beschreibung, z. B. "20 to 0.4 kV, compensated"
GROUNDMEAS_TIMEZONE = None       # Zeitzone der Geräteuhr, falls nicht UTC, z. B. "Europe/Berlin"
GROUNDMEAS_TIMESTAMP = None      # nur bei falsch gestellter Geräteuhr: Beginn der Aufzeichnung in UTC "JJJJ-MM-TT hh:mm:ss"

## 2 · Datei öffnen, Kanalrollen, Rohdaten-Diagnose

Prüfen: Sind alle Rollen richtig erkannt (Leiter-Erde-Spannungen, Fehlerstrom, Spulen, Station)? Ist der
richtige **Hauptkanal** des Fehlerstroms markiert (✓)? Meldet die Diagnose Sättigung, flache Kanäle, eine
abweichende Zeitbasis oder einen Zeitversatz zwischen Spannungs- und Stromkanälen?
Die Korrekturen der Kampagne werden hier angewendet und als Tabelle angezeigt.

In [ ]:
CAMPAIGN_FOLDER = WORK_FOLDER / LAYOUT.campaigns / CAMPAIGN
if DEMO and not CAMPAIGN_FOLDER.exists():
    demo.make_campaign(str(WORK_FOLDER), CAMPAIGN, layout=LAYOUT.language)
CFG = CAMPAIGN_FOLDER / FILE
SITE = sites.site_from_path(FILE)
CORR = corrections.load(CAMPAIGN_FOLDER) if APPLY_CORRECTIONS else corrections.Corrections.empty()
rec = gf.open_recording(str(CFG), fs_override=FS_OVERRIDE)
corrections_log = CORR.apply(rec)        # Wandlerfaktoren je Kanal – ab hier in allen Werten enthalten
roles = gf.detect_roles(rec.channel_names, rec.units, {**CORR.role_overrides(rec), **ROLE_OVERRIDES},
                        {**CORR.main_channel(rec), **MAIN_CHANNEL}, CORR.distances(rec))
MEASUREMENT = MEASUREMENT or CFG.stem
if len(corrections_log):
    display(Markdown("**Korrekturen der Kampagne** (in allen folgenden Werten enthalten)"))
    show(corrections_log)
for h in CORR.hints:
    print("⚠", h)
kind = "+".join(str(i18n.value_label("measurement_kind", k)) for k in roles.measurement_kind().split("+"))
print(rec, "| Start (cfg):", rec.t_start, "| Messart:", kind)
show(roles.table())
dg = diagnostics.diagnose(rec, roles)
print(dg.text())
show(dg.channels)

## 3 · Effektivwertverlauf – Auswertebereiche ablesen

Gleitender Echt-Effektivwert (20-ms-Fenster, DC je Fenster entfernt). Mit Zoom/Range-Slider die Zeiten ablesen:
**stationär** = nach dem Einschwingen und *vor* einer Wattreststromerhöhung/Spulenverstellung; **Vorfehler** und
**Nachfehler** = erdschlussfreier Betrieb (nach dem Abklingen der Verlagerungsspannung). Eingetragene Bereiche
werden farbig hinterlegt.

In [ ]:
INTERVALS = gf.Intervals(STEADY_STATE, PRE_FAULT, POST_FAULT)
interactive.rms_values(rec, roles, intervals=INTERVALS).show()
if SHOW_ENVELOPE:
    interactive.overview(rec, roles, intervals=INTERVALS).show()

## 4 · Auswertung

In [ ]:
res = None
if STEADY_STATE is None:
    display(Markdown("**➜ Zuerst `STEADY_STATE` (und `PRE_FAULT`/`POST_FAULT`) in Abschnitt 1 eintragen und erneut ausführen.**"))
else:
    res = gf.evaluate(rec, INTERVALS, SETTINGS, roles, alignment=ALIGNMENT, name=MEASUREMENT, z_fault_ohm=Z_FAULT,
                      metadata=dict(campaign=CAMPAIGN, site=SITE))
    print(res.summary())
    display(Markdown("**Bewertungsgrößen** – Messwert und Bewertung mit den Messunsicherheitszuschlägen aus Abschnitt 1"))
    show(res.table("assessment_quantities"))
    display(Markdown("**Plausibilitätsprüfungen**"))
    show(res.table("plausibility"))

### Spannungen vor, im und nach dem Fehler

In [ ]:
if res is not None and "voltage" in res.values:
    show(res.table("voltage_metrics"))
    interactive.voltage_spectrum(res).show()

### Fehlerstrom – Harmonische je Fenster und Statistik über den stationären Bereich

In [ ]:
if res is not None and res.roles.has("i_res"):
    interactive.window_spectrum(res).show()
    interactive.harmonics_time(res).show()
    if not res.table("coil_currents").empty:
        show(res.table("coil_currents"))
    show(res.table("statistics"))

### Station (Erdungs-/Berührungsspannung, Schirmstrom) und Trafo – falls gemessen

In [ ]:
if res is not None:
    for name in ("station_metrics", "station_voltages", "station_currents", "transformer_power"):
        if name in res.tables and not res.tables[name].empty:
            display(Markdown("**%s**" % report.table_title(name)))
            show(res.tables[name])

### Station: Erdungs- und Berührungsspannung je Frequenz

Je Sonde und Harmonische (I_F ≥ 1 % der Grundschwingung): **direkt** = Spannung/Fehlerstrom im Fehler, wie gemessen;
**fehlerbedingt** = nur der vom Fehlerstrom verursachte Anteil. Vor dem Fehler liegt an der Station oft schon eine
Spannung an (**Grundlast**, z. B. Ströme aus dem NS-Netz) – sie wird mit drei Verfahren abgezogen: Zeitbezug am
Fehlereintritt, Zeitbezug am Fehlerende und (falls gemessen) Bezug auf die Leiter-Leiter-Spannung.

**Wie verlässlich ist der fehlerbedingte Anteil?** `Streuung_Verfahren_proz` zeigt, wie gut die Verfahren
übereinstimmen. `Unsicherheit_proz` schätzt, wie stark die Grundlast das Ergebnis verfälschen kann: schwankende
Grundlast (Nulltest in den fehlerfreien Abschnitten) und – ohne Spannungsmessung – der Frequenzbezug über den
Fehlerstrom, jeweils × Grundlast/fehlerbedingter Anteil. Liegt eines davon über 20 %, steht in `unsicher` „True“ und
die Abbildungen zeigen den Wert als **⊗** – typisch bei 150 Hz, wo die Grundlast oft größer ist als der
fehlerbedingte Anteil. Die **Berührungsspannung** stammt von der 1-m-Sonde (U1m, UE1 …) oder – Näherung – von
der kürzesten Sonde bis 5 m (siehe Rollentabelle in Abschnitt 2). Für den Vergleich mit der Erdungsmessung gilt der
fehlerbedingte Anteil.

**PEN-Ströme und Ortsnetz** (falls `IPEN_v`/`IPEN_n` gemessen): Dieselbe Überlagerung trennt beim PEN-Strom die
Unsymmetrie der Last (schon vor dem Fehler vorhanden) vom fehlerbedingten Strom ins Ortsnetz I_ON. `IPEN_v` (Summe der
NS-Abgänge, vor dem Trafoknoten) und `IPEN_n` (zur Haupterdungsschiene) müssen denselben fehlerbedingten Strom führen
(Knotenregel – der Strom des Trafosternpunkts ändert sich nicht). **Z_E,Ortsnetz = ΔU_E/I_ON** je Frequenz und als
Effektivwert – für jede Erdungsspannungssonde und für das **Sondenmaximum** (`Kanal` = `max(UE)`, Spalte `Sonde`): je
Frequenz die höchste Spannung aller Sonden gegen die Stationserde, effektiv die Sonde mit dem höchsten Effektivwert.
Die Potentiale breiten sich unterschiedlich aus; jede Sonde misst höchstens die volle Erdungsspannung.
**Prüfung Parallelschaltung** (`Pruefung_Parallelschaltung`): Das Ortsnetz ist ein Zweig der Gesamterdung, also
|Z_E,Ortsnetz| ≥ |Z_E,Erdschluss| = |max(U_E)/I_F| (gleichbedeutend |I_ON| ≤ |I_F|), bei jeder Frequenz; mit
Schirmstrom zusätzlich gegen den Erdungsstrom I_E = I_F − I_S (`Pruefung_Erdungsstrom`). Verletzt erst, wenn die
Abweichung größer ist als die Messunsicherheit. Der Vergleich mit der Erdungsmessung folgt im nächsten Abschnitt.

In [ ]:
if res is not None and not res.tables.get("station_frequency", pd.DataFrame()).empty:
    show(res.table("station_frequency"))
    if LIMITS:
        display(Markdown("**Hochrechnung auf die Grenzwerte des Reststroms** (proportional; fehlerbedingt; mit Grundlast)"))
        show(res.table("station_extrapolation"))
    else:
        print("ℹ Hochrechnung auf die Grenzwerte des Reststroms übersprungen – LIMITS ist nicht gesetzt (Abschnitt 1).")
    interactive.station_frequency(res).show()
    if not res.tables.get("station_currents_frequency", pd.DataFrame()).empty:
        display(Markdown("**PEN- und Schirmströme je Frequenz** – im Fehler gemessen, Grundlast (Unsymmetrie der Last) und fehlerbedingter Anteil (Anteil = ΔI/I_F)"))
        show(res.table("station_currents_frequency"))
    if not res.tables.get("station_lv_network", pd.DataFrame()).empty:
        display(Markdown("**Ortsnetz: Strom I_ON und Erdungsimpedanz Z_E,Ortsnetz = ΔU_E/I_ON** (je Frequenz und effektiv; Prüfung Parallelschaltung: Z_E,Ortsnetz ≥ max(U_E)/I_F, Prüfung Erdungsstrom: |I_ON| ≤ |I_E|)"))
        show(res.table("station_lv_network"))
        show(res.table("station_lv_network_eff"))
        interactive.lv_network(res.table("station_lv_network")).show()
elif res is not None:
    print("Keine Erdungs-/Berührungsspannung gemessen – Abschnitt entfällt.")

### Station: Vergleich mit der Erdungsmessung (groundmeas)

Die Erdungsmessungen (Strom-Einspeisung) liegen in der **groundmeas-Datenbank**. Sie wird hier nur **gelesen**.
Der Ort kommt aus der Messorte-Datei der Kampagne, sonst aus dem Ordnernamen (Tippfehler werden erkannt) – bei Bedarf
`GROUNDMEAS_LOCATION` bzw. `GROUNDMEAS_MEASUREMENTS` in Abschnitt 1 setzen. Verglichen wird **Spannung je Ampere**
(V/A = Ω) bei derselben Frequenz: Einspeisung (Sonde in vergleichbarem Abstand) ↔ fehlerbedingter Anteil im
Erdschluss. Für die Erdungsspannung an einer fernen Sonde ist das Verhältnis der wirksame **Reduktionsfaktor**
(Rückstrom über Kabelschirme). Sondenabstände, die nicht im Kanalnamen stehen (`UEr`, `UE`), in der Korrekturen-Datei
(Spalte `Abstand_m`) nachtragen. Der Eintrag dieser Messung in groundmeas folgt in Abschnitt 7.

**Reduktionsfaktor** r = Z_E,Erdschluss/Z_E,Erdungsmessung (Erdungsspannung je Ampere, nur Beträge): je Harmonische
und als Effektivwert über alle Harmonischen (gewichtet mit dem Fehlerstrom); zum Vergleich der Effektivwert aus den
gemessenen Effektivwerten (mit Grundlast, Erdungsimpedanz bei 50 Hz) und – falls gemessen – die Gegenprobe aus dem
Schirmstrom. Maßgeblich ist das **Sondenmaximum** des Erdschlusses (`max(UE)`: höchste Spannung aller Sonden) gegen die
fernste Sonde der Erdungsmessung; dazu jede Sonde gegen die Sonde mit vergleichbarem Abstand. Fehlt die Erdungsmessung
bei einer Frequenz, wird zwischen den Nachbarfrequenzen interpoliert (nie extrapoliert).
**Einordnung:** r bei 50 Hz gegen den Bereich, den Sie erwarten (`R_TYPICAL` in Abschnitt 1, z. B. aus eigener
Erfahrung mit vergleichbaren Anlagen – groundfault hat keinen vorgegebenen Bereich); mit steigender Frequenz sollte r
fallen (induktive Kopplung der Rückleiter, f → ∞: r → 0). Deutlich kleinere Werte deuten auf einen zu hohen Messwert
der Erdungsmessung (methodischer Fehler des Strom-Spannungs-Verfahrens) oder weitere Rückleiter.
**Bedeutung für die Bewertung:** Bei kleinen Impedanzen wirken relative Abweichungen groß – entscheidend ist die
Spannung. Beide Impedanzen werden auf den Bezugsstrom (`REFERENCE_CURRENT`, sonst der größte Wert von `LIMITS`)
umgerechnet und die Abweichung auf F·U_Tp (`F_FACTOR`, `U_TP`) bezogen: `Bewertungsrelevanz` „vernachlässigbar“
(< 10 %), „Bewertung unverändert“ (beide auf derselben Seite von F·U_Tp) oder „bewertungsrelevant“. Ohne diese
eigenen Werte wird die Bewertungsrelevanz nicht bestimmt.
**Ortsnetz ↔ Erdungsmessung:** Als Parallelzweig sollte Z_E,Ortsnetz auch über Z_E der Erdungsmessung liegen
(`Vergleich_Erdungsmessung`). Ist es kleiner, die Prüfung gegen den Erdschluss aber erfüllt, misst die Erdungsmessung
vermutlich zu viel – ein Befund, kein Fehler der Ortsnetzauswertung.

In [ ]:
COMPARISON = RATIO = None
RF = RF_EFF = LV_CHECK = LV_CHECK_EFF = pd.DataFrame()
if res is not None and not res.tables.get("station_frequency", pd.DataFrame()).empty:
    try:
        DB = earthing.Database(GROUNDMEAS_DB)          # nur lesend geöffnet
    except (FileNotFoundError, ValueError) as ex:
        DB = None
        print("ℹ groundmeas-Datenbank nicht verfügbar:", ex)
    if DB is not None:
        try:
            ENTRY = sites.load(CAMPAIGN_FOLDER).for_site(SITE)      # Zeile der Messorte-Datei
        except ValueError as ex:
            ENTRY = None
            print("⚠", ex)
        LOCATION = GROUNDMEAS_LOCATION or (ENTRY.location if ENTRY is not None and ENTRY.location else None) or \
            earthing.location_from_site(SITE)
        CHOICE = DB.choose_location(LOCATION, exact_only=bool(GROUNDMEAS_LOCATION or (ENTRY is not None and ENTRY.location)),
                                    preset='GROUNDMEAS_LOCATION = "{}"')
        LOCATION_DB = CHOICE["location"]
        if CHOICE["note"]:
            print("ℹ", CHOICE["note"])
        if LOCATION_DB is None and len(CHOICE["candidates"]):
            display(Markdown("**Orte in groundmeas, die ähnlich heißen:**"))
            show(CHOICE["candidates"])
        if GROUNDMEAS_MEASUREMENTS:
            MEAS = DB.measurements()[lambda d: d["measurement_id"].isin(GROUNDMEAS_MEASUREMENTS)]
        else:
            MEAS = DB.measurements(location=LOCATION_DB) if LOCATION_DB else DB.measurements().iloc[0:0]
        MEAS = earthing.exclude_own(MEAS, CAMPAIGN, MEASUREMENT)   # eigenen Eintrag nicht mit sich selbst vergleichen
        display(Markdown("**groundmeas: %s** – %d Messung(en) am Ort „%s“ (gesucht: „%s“)"
                         % (DB.path, len(MEAS), LOCATION_DB or "–", LOCATION)))
        show(MEAS)
        RESPONSE = earthing.frequency_response(DB, MEAS["measurement_id"])
        COMPARISON, RATIO = earthing.compare(res.table("station_frequency"), RESPONSE, measurement=MEASUREMENT)
        if len(RATIO):
            display(Markdown("**Verhältnis Erdschluss / Erdungsmessung je Frequenz** (Spalte `Abstandsvergleich`: „abweichend“ = Sonden nicht vergleichbar, nicht als Reduktionsfaktor deuten)"))
            show(RATIO)
        if COMPARISON["source"].isin([earthing.SOURCE_INJECTION, earthing.SOURCE_GROUNDMEAS_FAULT]).any():
            interactive.earthing_comparison(COMPARISON, RATIO).show()
        else:
            print("Keine passenden Werte in groundmeas für „%s“ – Ort/IDs in Abschnitt 1 prüfen." % LOCATION)
        RF, RF_EFF = earthing.reduction_factor(res, RESPONSE, reference_current_A=REFERENCE_CURRENT,
                                               r_typical=R_TYPICAL)      # U_Tp, F und LIMITS aus SETTINGS
        if len(RF_EFF) and R_TYPICAL is None:
            print("ℹ Keine Einordnung von r bei 50 Hz – R_TYPICAL ist nicht gesetzt (Abschnitt 1).")
        NOT_SET = [v for v, c in (("REFERENCE_CURRENT oder LIMITS", "reference_current_A"), ("U_TP", "U_Tp_V"),
                                  ("F_FACTOR", "F_factor")) if len(RF_EFF) and RF_EFF[c].isna().all()]
        if NOT_SET:                    # Bedeutung für die Bewertung: Bezugsstrom, U_Tp und F aus Abschnitt 1
            print("ℹ Bedeutung der Abweichung für die Bewertung nicht bestimmt – nicht gesetzt (Abschnitt 1): %s"
                  % ", ".join(NOT_SET))
        if len(RF_EFF):
            display(Markdown("**Reduktionsfaktor** – effektiv (`r_eff`, zum Vergleich aus den gemessenen Effektivwerten und aus dem Schirmstrom; Einordnung bei 50 Hz, Verlauf über f, Bedeutung für die Bewertung) und je Harmonische (`r`); erste Zeilen: Sondenmaximum"))
            show(RF_EFF); show(RF)
            interactive.reduction_factor(RF, RF_EFF).show()
        LV_CHECK, LV_CHECK_EFF = earthing.check_lv_network(res, RESPONSE)
        if len(LV_CHECK):
            display(Markdown("**Ortsnetz ↔ Erdungsmessung** (Verhältnis Z_E,Ortsnetz/Z_E ≥ 1 erwartet; Prüfung Parallelschaltung: gegen den Erdschluss)"))
            show(LV_CHECK_EFF); show(LV_CHECK)
            interactive.lv_network(res.table("station_lv_network"), LV_CHECK).show()
        for h in earthing.reduction_hints(RF_EFF, LV_CHECK):
            print("⚠", h)
        DB.close()

## 5 · Signal-Explorer – Werte an der Kurve prüfen

Alle Kanäle der Aufzeichnung: **oben** der Effektivwert je 200-ms-Fenster über der ganzen Datei – der Mouse-over zeigt
für alle gewählten Kanäle Effektivwert, Harmonische (H1, H3, H5, H7), THD und Gleichanteil zum selben Zeitpunkt;
**darunter** Momentanwerte, Spektrum und Kennwerte des grau markierten Ausschnitts. Bedienung: Kanäle links wählen
(Strg/⌘ bzw. Shift für mehrere), Ausschnitt mit dem Schieberegler, *Zeitpunkt* (Fehlereintritt, Max-Fenster, …)
oder ◀ ▶ verschieben, *Breite* und *bis Ordnung* einstellen. Das Max-Fenster liefert dieselben Werte wie die Auswertung.
Der Explorer funktioniert auch schon vor dem Eintragen der Bereiche (dann ohne Max-Fenster).

In [ ]:
EX = explorer.explorer(rec, res, alignment=ALIGNMENT) if SIGNAL_EXPLORER else None
EX

## 6 · Kontrollen (optional)

In [ ]:
FT = VC = None
if res is not None:
    interactive.frequency_diagnosis(rec, res).show()
    P_RMS = CAMPAIGN_FOLDER / RMS_FILE if RMS_FILE else None
    if P_RMS is not None and not P_RMS.is_file():
        CANDIDATES = sorted(str(p.relative_to(CAMPAIGN_FOLDER)) for p in P_RMS.parent.glob("*.cfg")) if P_RMS.parent.is_dir() else []
        print("⚠ RMS_FILE nicht gefunden: %s – Kontrolle übersprungen.%s" % (
            RMS_FILE, (" Im Ordner vorhanden: " + ", ".join(CANDIDATES)) if CANDIDATES else ""))
    elif P_RMS is not None:
        rec_rms = gf.open_recording(str(P_RMS))
        CORR.apply(rec_rms)                 # dieselben Korrekturen auch auf die RMS-Datei
        for h in CORR.hints:                # Hinweise zur RMS-Datei (z. B. kein Vorzeichen)
            print("⚠", h)
        display(Markdown("**Eigener Echt-Effektivwert ↔ RMS des Messgeräts** – Abweichungen entstehen v. a. durch DC-Offsets der Zangen (das Gerät rechnet den DC mit, die Auswertung entfernt ihn)."))
        show(rms.compare_with_instantaneous(rec, rec_rms, STEADY_STATE))
    if V_CURVE and res.roles.has("i_res"):
        FT = rms.fundamental_trend(rec, STEADY_STATE, res.roles)
        VC = rms.v_curve(FT["t_s"], FT["IH1_A"])
        print(i18n.format_text("V-Kurve: kleinster Grundschwingungsstrom {:.2f} A bei {:.2f} s{} {}",
                               VC.I_min_measured, VC.t_min_s, "" if np.isnan(VC.I_min_fit) else i18n.format_text(
                                   " (Ausgleichsparabel: {:.2f} A)", VC.I_min_fit), VC.note))
        show(FT)
        report.v_curve(FT["t_s"], FT["IH1_A"], VC, quantity="I_G")

## 7 · Export für Kampagnenauswertung und Bericht

* `2_Auswertung/<Messung>/<Messung>_ergebnis.json` – vollständiges Ergebnis (Grundlage für `Kampagne_auswerten.ipynb`)
* `2_Auswertung/<Messung>/tabellen/` – alle Tabellen als CSV (Semikolon/Dezimalkomma), Markdown, eine Excel-Mappe
  und eine Word-Datei mit formatierten Tabellen
* `3_Abbildungen/<Messung>/` – Abbildungen in Endgröße (15,9 cm breit, ohne Titel) als PDF, SVG und PNG
* `3_Abbildungen/<Messung>/<Messung>_explorer.html` – Signal-Explorer zum Weitergeben (im Browser öffnen, ohne Python;
  bei Stationen mit den Abbildungen „frequenzselektiv“ und „Vergleich mit der Erdungsmessung“)

In [ ]:
if res is not None:
    RESULTS = CAMPAIGN_FOLDER / LAYOUT.results / MEASUREMENT
    FIGURES = CAMPAIGN_FOLDER / LAYOUT.figures / MEASUREMENT
    p = res.save_json(LAYOUT.result_path(CAMPAIGN_FOLDER, MEASUREMENT))
    has_comparison = COMPARISON is not None and COMPARISON["source"].isin(
        [earthing.SOURCE_INJECTION, earthing.SOURCE_GROUNDMEAS_FAULT]).any()
    TABLES = dict(res.tables)
    if has_comparison:                 # Vergleich mit groundmeas in dieselbe Excel-/Word-Datei
        TABLES.update(earthing_comparison=COMPARISON, earthing_ratio=RATIO)
    TABLES.update({n: df for n, df in (("reduction_factor", RF), ("reduction_factor_eff", RF_EFF),
                                       ("lv_network_check", LV_CHECK), ("lv_network_check_eff", LV_CHECK_EFF)) if len(df)})
    tables = report.export_tables(TABLES, str(RESULTS / ("tables" if LAYOUT.language == "en" else "tabellen")),
                                  prefix=MEASUREMENT + "_", formats=("csv", "md"))
    figures = report.all_figures(res, str(FIGURES), rec=rec, prefix=MEASUREMENT + "_", formats=EXPORT_FORMATS)
    if VC is not None:
        figures += report.save(report.v_curve(FT["t_s"], FT["IH1_A"], VC, quantity="I_G"),
                               str(FIGURES / (MEASUREMENT + "_v_curve")), EXPORT_FORMATS, close=True)
    extra = {}
    if not res.tables.get("station_frequency", pd.DataFrame()).empty:
        extra["Erdungs- und Berührungsspannung frequenzselektiv"] = interactive.station_frequency(res)
    if has_comparison:
        figures += report.save(report.earthing_comparison(COMPARISON, RATIO),
                               str(FIGURES / (MEASUREMENT + "_earthing_comparison")), EXPORT_FORMATS, close=True)
        extra["Erdungsmessung (groundmeas) und Erdschluss je Frequenz"] = interactive.earthing_comparison(COMPARISON, RATIO)
    if len(RF_EFF):
        figures += report.save(report.reduction_factor(RF, RF_EFF), str(FIGURES / (MEASUREMENT + "_reduction_factor")),
                               EXPORT_FORMATS, close=True)
        extra["Reduktionsfaktor je Harmonische"] = interactive.reduction_factor(RF, RF_EFF)
    if not res.tables.get("station_lv_network", pd.DataFrame()).empty:
        if len(LV_CHECK):              # mit Erdungsmessung (sonst ohne – Standardabbildung station_lv_network)
            figures += report.save(report.lv_network(res.table("station_lv_network"), LV_CHECK),
                                   str(FIGURES / (MEASUREMENT + "_lv_network_injection")), EXPORT_FORMATS, close=True)
        extra["Ortsnetz: Strom und Erdungsimpedanz"] = interactive.lv_network(
            res.table("station_lv_network"), LV_CHECK if len(LV_CHECK) else None)
    if EXPLORER_HTML:
        h = explorer.explorer_html(rec, str(FIGURES / (MEASUREMENT + "_explorer.html")), res, figures=extra,
                                   alignment=ALIGNMENT)
        print("Signal-Explorer (HTML):", Path(h).relative_to(WORK_FOLDER))
    if RECORD_INTERVALS and DEMO:      # die Demo ändert nichts außerhalb ihres Kampagnenordners
        print("Messregister des Arbeitsordners unverändert (Demo)")
    elif RECORD_INTERVALS:
        args = (str(WORK_FOLDER), str(CFG.relative_to(WORK_FOLDER)), STEADY_STATE, PRE_FAULT, POST_FAULT, Z_FAULT)
        ok = register.record_intervals(*args)
        if not ok:                     # neue Datei → Register neu einlesen (Handspalten bleiben erhalten)
            register.write_register(str(WORK_FOLDER))
            ok = register.record_intervals(*args)
        print("Messregister aktualisiert" if ok else "Datei nicht im Messregister – Pfad prüfen")
    print("Ergebnis:", Path(p).relative_to(WORK_FOLDER))
    print("%d Tabellen-Dateien, %d Abbildungsdateien" % (len(tables), len(figures)))

### groundmeas – diese Messung eintragen

Die ausgewertete Messung kann als **Erdschlussmessung** (`staged_fault_test`) in die groundmeas-Datenbank übernommen
werden – dort steht sie im Dashboard am selben Ort wie die Erdungsmessung. **Station:** Fehlerstrom, Erdungs- und
Berührungsspannung je Sonde (mit Abstand) und Frequenz – fehlerbedingter Anteil, Winkel bezogen auf I_F – und
Schirmstrom; **UW:** Fehlerstrom je Ordnung, Fehlerwiderstand = Zusatzimpedanz. Ort, Koordinaten, Anlagentyp,
Spannungsebene und Entfernung zum UW stehen in der Messorte-Datei der Kampagne (wird beim ersten Mal aus groundmeas
vorbelegt angelegt – für einen **neuen Ort Koordinaten eintragen**). Zunächst wird nur angezeigt, was eingetragen würde
(Trockenlauf); geschrieben wird mit `GROUNDMEAS_EXPORT = True` – nach einer Sicherungskopie der Datenbank, in einer
Transaktion, ohne vorhandene Einträge zu verändern. Eine schon eingetragene Messung wird erkannt;
`GROUNDMEAS_EXPORT = "replace"` erneuert den eigenen Eintrag.

In [ ]:
legacy.check_option("GROUNDMEAS_EXPORT", GROUNDMEAS_EXPORT, (False, True, "replace"))
PLAN = None
if res is not None:
    try:
        PLAN = earthing.export_to_groundmeas(
            res, CAMPAIGN_FOLDER, GROUNDMEAS_DB, location=GROUNDMEAS_LOCATION, write=bool(GROUNDMEAS_EXPORT),
            existing="replace" if GROUNDMEAS_EXPORT == "replace" else "skip", uncertain_mode=GROUNDMEAS_UNCERTAIN,
            description=GROUNDMEAS_DESCRIPTION, timezone=GROUNDMEAS_TIMEZONE, timestamp=GROUNDMEAS_TIMESTAMP,
            units=GROUNDMEAS_UNITS)
    except (FileNotFoundError, ValueError) as ex:
        print("ℹ groundmeas-Eintrag nicht möglich:", ex)
if PLAN is not None:
    for h in PLAN["hints"]:
        print("ℹ", h)
    for w in PLAN["warnings"]:
        print("⚠ vor dem Eintragen prüfen:", w)
    show(earthing.plan_overview({MEASUREMENT: PLAN}))
    if PLAN["written"]:
        print("✓ in groundmeas %s: Messung %d am Ort „%s“ (Ort %d) – Sicherungskopie: %s – jetzt GROUNDMEAS_EXPORT wieder auf False stellen."
              % ("erneuert" if PLAN.get("renewed") else "eingetragen", PLAN["measurement_id"], PLAN["location"],
                 PLAN["location_id"], PLAN["backup"]))
    elif not PLAN["writable"]:
        print("ℹ Nicht eingetragen: %s" % ("; ".join(PLAN["reasons"]) or PLAN.get("reason", "")))
    else:
        display(Markdown("**Trockenlauf** – so würde die Messung eingetragen (schreiben mit `GROUNDMEAS_EXPORT = True`):"))
    print("Beschreibung:", PLAN["measurement"]["description"])
    show(earthing.plan_items(PLAN))

---
**Konventionen**
* Phasoren sind Effektivwertzeiger (Kosinus-Konvention). Winkel in den Tabellen „bez. U_L1-L2“ sind auf die
  Grundschwingung der Leiter-Leiter-Spannung U_L1-L2 bezogen (Ordnung m: X_m·e^(−j·m·φ)) und damit unabhängig vom
  Zeitursprung der Aufzeichnung.
* Polarität: Der Wirkanteil der Grundschwingung muss positiv sein (Iw > 0). Ist er negativ, wird der Fehlerstrom um
  180° gedreht (Hinweis in der Tabelle). Mit dieser Orientierung bedeutet **Iv > 0 Überkompensation** (induktiver
  Überschuss).
* DC-Offset: je Fenster per Median entfernt. Effektivwert = Echt-Effektivwert über alle Frequenzen, nicht nur über
  die ausgewählten Harmonischen.